# Week 1 -churn Prediction Pipeline
## Day2 - Preprocessing, SMOTE and Model Training

**Goal:** Clean data. encode features. balance classes. train 3 models. cross validate

**Input:** WA_Fn-UseC_-Telco-Customer-churn.csv

**Output:** Trained XGBoost model saved as model.pkl

**Author:** Martin James |github.com/M20Jay

## Section 1 - Import Libraries
We import all tools needed for preprocessing, balancing and model training

In [64]:
# Import libraries
import sys
!{sys.executable} -m pip install psycopg2-binary
import pandas as pd
import psycopg2
import numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings("ignore")

## Section 2- Load and Prepare Data
We load dataset from Day 1 and apply initial cleaning before preprocessing

In [65]:
# total_charges to Database
conn =psycopg2.connect(
    dbname="churn_db",
    user="martinjames",
    host="localhost"
)

# Pull data from churn raw
df = pd.read_sql("SELECT * FROM churn_raw", conn)
conn.close()


# Fix total_charges — convert from object to numeric
df['total_charges']= pd.to_numeric(df['total_charges'], errors= 'coerce')

# Drop rows with missing total_charges (only 11 rows)
print("Missing total_charges:", df['total_charges'].isnull().sum())
df.dropna(subset=['total_charges'],inplace =True)

#Encode target column with  - Yes = 1 , No =0
df['churn']=df['churn'].map({'Yes':1, 'No':0})

print("\nShape after cleaning:", df.shape)
print("\nchurn value counts:")
print(df['churn'].value_counts())

Missing total_charges: 0

Shape after cleaning: (7043, 23)

churn value counts:
churn
0    5174
1    1869
Name: count, dtype: int64


## Section 3 — Feature Engineering
We recreate the 3 features from Day 1 and drop columns not needed for modelling.

In [66]:
#  Feature Engineering

# Recreate features from Day 1
# tenure_group encoding:
# 0 = New   (0-12 months)
# 1 = Mid   (12-24 months)
# 2 = Loyal (24-72 months)

df['tenure_group'] = pd.cut(df['tenure'],
    bins=[0, 12, 24, 72],
    labels=[0, 1, 2])

df['tenure_group'] = df['tenure_group'].fillna(0).astype(int)


df['charges_per_month'] = df['total_charges'] / (df['tenure'] + 1)

df['is_high_value'] = (df['monthly_charges'] > 70).astype(int)

# Drop columns not needed for modelling
df.drop(columns=['customer_id', 'loaded_at', 'id'], inplace=True)
print("Features after engineering:")
print(df.columns.to_list())
print("\nShape :", df.shape)


Features after engineering:
['gender', 'senior_citizen', 'partner', 'dependents', 'tenure', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn', 'tenure_group', 'charges_per_month', 'is_high_value']

Shape : (7043, 23)


## Section 4 -Encode Categorical Variables
We convert all text columns to numbers so the model can process them

In [67]:
# Ecode categorical variables
le =LabelEncoder()

# Columns to encode
df.select_dtypes(include='object').columns.tolist()

# Convert tenure_group from category to string for encoding
df['tenure_group'] = df['tenure_group'].astype(str)

# Columns to encode
cat_cols = [
    'gender', 
    'partner', 
    'dependents', 
    'phone_service', 
    'multiple_lines', 
    'internet_service', 
    'online_security', 
    'online_backup', 
    'device_protection', 
    'tech_support', 
    'streaming_tv', 
    'streaming_movies', 
    'contract', 
    'paperless_billing', 
    'payment_method'
]

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))



print("Encoding complete ✅")
print(df.dtypes)


Encoding complete ✅
gender                 int64
senior_citizen         int64
partner                int64
dependents             int64
tenure                 int64
phone_service          int64
multiple_lines         int64
internet_service       int64
online_security        int64
online_backup          int64
device_protection      int64
tech_support           int64
streaming_tv           int64
streaming_movies       int64
contract               int64
paperless_billing      int64
payment_method         int64
monthly_charges      float64
total_charges        float64
churn                  int64
tenure_group          object
charges_per_month    float64
is_high_value          int64
dtype: object


## Section 5 -Train Test Split
We separate features from target and split into training and test sets

In [68]:
# Train Test Split

#Separate features and target
X= df.drop(columns=['churn'])
y=df['churn']
# Split 80% train 20% test — stratified to preserve churn ratio
X_train,X_test,y_train,y_test= train_test_split(X,y, test_size=0.2,random_state=42, stratify=y)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\n churn ratio in training set:")
print(y_train.value_counts(normalize=True).round(3)*100)


Training set: (5634, 22)
Test set: (1409, 22)

 churn ratio in training set:
churn
0    73.5
1    26.5
Name: proportion, dtype: float64


## Section 6 - SMOTE Application
We apply SMOTE to balance training set. SMOTE creates synthetic minority class samples so the model learns churners as well as non-churners

In [69]:
# SMOTE application
print("Before SMOTE:", y_train.value_counts())
smote=SMOTE(random_state=42)
X_train_bal,y_train_bal =smote.fit_resample(X_train,y_train)


print("\nAfter SMOTE:")
print(pd.Series(y_train_bal.value_counts()))
print("\n New training size:", X_train_bal.shape)



Before SMOTE: churn
0    4139
1    1495
Name: count, dtype: int64

After SMOTE:
churn
0    4139
1    4139
Name: count, dtype: int64

 New training size: (8278, 22)


## Section 7 - Train 3 Models
We train Logistic Regression, Random Forest and XGBoost on the balanced dataset.
We use cross validation to get more reliable scores.

In [70]:
#  Train 3 models
# Force all features to numeric for XGBoost
X_train_bal = X_train_bal.astype(float)
X_test = X_test.astype(float)

#Define models
models ={
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state =42),
    'Random Forest': RandomForestClassifier(n_estimators =100, random_state =42),
    'XGBoost' : XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
    }

#cross validate each model
cv=StratifiedKFold(n_splits=5, shuffle = True, random_state=42)

print("=== Cross validation results ===")
for name, model in models.items():
    scores = cross_val_score(model,X_train_bal,y_train_bal, cv=cv, scoring= 'roc_auc')
    print(f"{name}: AUC ={scores.mean():.4} +/- {scores.std():.4}")


=== Cross validation results ===
Logistic Regression: AUC =0.8851 +/- 0.004396
Random Forest: AUC =0.9283 +/- 0.004693
XGBoost: AUC =0.9283 +/- 0.003486


## Section 8- Final Model Training and Evaluation



In [71]:
# Train XGBoost
xgb_model= XGBClassifier(n_estimators =100, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_bal,y_train_bal)

# Predict on Test
y_pred=xgb_model.predict(X_test)
y_pred_proba =xgb_model.predict_proba(X_test)[:,1]


# Evaluate
print("=== XGBoost test results ===")
print(f" AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\n Classification report:")
print(classification_report(y_test, y_pred))
print("\nConfusion matix")
print(confusion_matrix(y_test, y_pred))


=== XGBoost test results ===
 AUC: 0.8159

 Classification report:
              precision    recall  f1-score   support

           0       0.86      0.83      0.84      1035
           1       0.57      0.61      0.59       374

    accuracy                           0.77      1409
   macro avg       0.71      0.72      0.72      1409
weighted avg       0.78      0.77      0.78      1409


Confusion matix
[[861 174]
 [146 228]]


## Section 9 — Save Model
We save the trained XGBoost model to disk so it can be loaded by the Flask API in Day 3

In [72]:
# Save Model
joblib.dump(xgb_model,'../src/churn_model.pkl')
print("Model saved to src/churn_model.pkl ✅")

Model saved to src/churn_model.pkl ✅


In [4]:
# Load model
model = joblib.load('../src/churn_model.pkl')
print(type(model))

<class 'xgboost.sklearn.XGBClassifier'>
